# 01_data: build the training data

**Purpose:** turn raw clicks into one row per session: the click words in order, and a label (1 = bought, 0 = didn't).

**Input:** REES46 multi-category store, October 2019 (attached on Kaggle). One row = one click.

**Output (`data/sample/`):**
- `events_sample.parquet`: the raw clicks of ~1 in 150 users (all their sessions). Made by Step 1 on Kaggle.
- `sessions_sample.parquet`: one row per session: `tokens` (e.g. `["view_expensive_start", "view_expensive_again_quick"]`), `label`, `user_id`, `start_time`, `n_clicks`, `cart_value`. Made by Steps 2-3, which run anywhere from the repo.

**Findings (20 Nov):** no `remove_from_cart` in October; 6.5% of sessions buy; only 44% of buyers add to cart first.
So each click word also carries price band, "again" (same product seen before) and time since previous click.

**Owner:** TBD (Data role)


## Step 1: read the month in pieces, count click types, keep a sample of users
The file is ~5 GB, so we read 5 million rows at a time. We count every click type over the **whole** month
(to see if `remove_from_cart` exists) and keep only users whose id is divisible by 150 (~0.7% of users).
Sampling **by user** keeps all of a user's sessions together, so later we can split train/test by user without leaks.

In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print("\n".join(files))
MONTH_FILE = [f for f in files if "2019-Oct" in f][0]

KEEP_ONE_IN = 150
COLS = ["event_time", "event_type", "product_id", "category_code", "brand", "price", "user_id", "user_session"]

event_counts = pd.Series(dtype="int64")
parts = []
for chunk in pd.read_csv(MONTH_FILE, usecols=COLS, chunksize=5_000_000):
    event_counts = event_counts.add(chunk["event_type"].value_counts(), fill_value=0)
    parts.append(chunk[chunk["user_id"] % KEEP_ONE_IN == 0])

events = pd.concat(parts, ignore_index=True).dropna(subset=["user_session"])
print("\nClick types in the WHOLE month:\n", event_counts.astype(int))
print("\nremove_from_cart exists:", "remove_from_cart" in event_counts.index)
print("Sampled clicks:", len(events), "| users:", events.user_id.nunique(), "| sessions:", events.user_session.nunique())

## Step 2 (runs from the repo, not on Kaggle): one row per session, cut before the first purchase
Uses `build_sessions` from `nudge/contract.py`, the same code the demo app uses to turn clicks into words.
Each session is sorted by time; if it contains a purchase, only the clicks **before** the first purchase are kept and
the label is 1 (the model must never see the purchase). Each click becomes one word:
type (view/cart) + price band (cheap <€100, mid, expensive ≥€250) + `again` if the product was seen before + time since previous click (start / quick <20 s / normal 20–70 s / long 70 s–30 min / away >30 min).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so we can import nudge
import pandas as pd
from nudge.contract import build_sessions

events = pd.read_parquet("../data/sample/events_sample.parquet")
sessions = build_sessions(events)
print(sessions.head())

## Step 3: the numbers that shape the whole project
- **Share that buy**: if it is small (e.g. 5%), a model that always says "won't buy" gets 95% accuracy and is useless, so we report precision, recall and F1.
- **Session length**: tells us how long the model's input must be.

In [ ]:
print("Sessions:", len(sessions))
print("Share that buy:", round(sessions.label.mean(), 4))
print("Clicks per session: median", sessions.n_clicks.median(), "| 90th percentile", sessions.n_clicks.quantile(0.9),
      "| max", sessions.n_clicks.max())
print("Share that buy, among sessions with a cart:", round(sessions[sessions.cart_value > 0].label.mean(), 4))
again = sessions.tokens.apply(lambda t: any("_again" in w for w in t))
print("Buy rate if a product was viewed again:", round(sessions[again].label.mean(), 3), "| if not:", round(sessions[~again].label.mean(), 3))
print(sessions.tokens.explode().value_counts().head(10))

## Step 4: save (small files that go into the repo's `data/sample/`)

In [ ]:
sessions.to_parquet("../data/sample/sessions_sample.parquet", index=False)
print("saved", len(sessions), "sessions")